# Flash Attention: Memory-Efficient Attention with Tiling

This notebook explores **Flash Attention**, a groundbreaking algorithm that makes transformer attention faster and more memory-efficient. We'll understand why standard attention is slow, how Flash Attention solves this, and implement a simplified version from scratch.

**What you'll learn:**
- Why standard attention is memory-bound (not compute-bound)
- The GPU memory hierarchy and why it matters
- The Flash Attention algorithm: tiling + online softmax
- Implementing simplified Flash Attention from scratch
- Memory and performance comparison


## Setup


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
from aiml_notebooks import get_device, set_seed

# Set style and seed for reproducibility
sns.set_style('whitegrid')
set_seed(42)
device = get_device()


## Part 1: The Problem with Standard Attention

### Standard Attention Review

The standard scaled dot-product attention computes:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

For a sequence of length $N$ with dimension $d$:
- $Q, K, V \in \mathbb{R}^{N \times d}$
- Attention matrix $S = QK^T \in \mathbb{R}^{N \times N}$
- Memory: $O(N^2)$ to store the attention matrix


Compare the causal Flash Attention implementation against the standard masked attention baseline.

In [ ]:
def standard_attention(Q, K, V):
    """
    Standard scaled dot-product attention.
    This is the naive implementation that materializes the full N×N attention matrix.
    
    Args:
        Q: Queries (batch, seq_len, d_k)
        K: Keys (batch, seq_len, d_k)
        V: Values (batch, seq_len, d_k)
    
    Returns:
        output: Attention output (batch, seq_len, d_k)
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores (N×N matrix!)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Step 2: Apply softmax (row-wise)
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 3: Weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Test with small sequence
batch_size = 2
seq_len = 8
d_k = 16

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, weights = standard_attention(Q, K, V)
print(f"Input shapes: Q={Q.shape}, K={K.shape}, V={V.shape}")
print(f"Attention weights shape: {weights.shape} (N×N matrix)")
print(f"Output shape: {output.shape}")


### The Memory Bottleneck

The problem: As sequence length grows, memory usage explodes!

| Sequence Length | Attention Matrix Size | Memory (fp32) |
|----------------|----------------------|---------------|
| 512            | 512 × 512            | 1 MB          |
| 2,048          | 2,048 × 2,048        | 16 MB         |
| 8,192          | 8,192 × 8,192        | 256 MB        |
| 32,768         | 32,768 × 32,768      | 4 GB          |

And this is **per head, per layer, per batch element**!


In [ ]:
def compute_attention_memory(seq_len, num_heads=1, batch_size=1, dtype_bytes=4):
    """Compute memory needed for attention matrix in bytes."""
    # Attention matrix is N×N per head per batch element
    return batch_size * num_heads * seq_len * seq_len * dtype_bytes

# Visualize memory scaling
seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
memory_mb = [compute_attention_memory(n) / (1024**2) for n in seq_lengths]

plt.figure(figsize=(10, 6))
plt.semilogy(seq_lengths, memory_mb, 'o-', linewidth=2, markersize=8, color='#e74c3c')
plt.axhline(y=16000, color='gray', linestyle='--', label='16 GB (A100 memory)', alpha=0.7)
plt.axhline(y=24000, color='green', linestyle='--', label='24 GB (RTX 4090 memory)', alpha=0.7)
plt.xlabel('Sequence Length', fontsize=12)
plt.ylabel('Memory for Attention Matrix (MB)', fontsize=12)
plt.title('Attention Memory Scaling: O(N²) is Brutal', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Memory requirements for single attention matrix (fp32):")
for n, mem in zip(seq_lengths, memory_mb):
    print(f"  N={n:>6}: {mem:>10.2f} MB")


## Part 2: GPU Memory Hierarchy

### Why Standard Attention is I/O Bound

Modern GPUs have a memory hierarchy:

| Memory Type | Size | Bandwidth | Access Speed |
|-------------|------|-----------|-------------|
| **SRAM** (on-chip) | ~20 MB | ~19 TB/s | Very fast |
| **HBM** (off-chip) | ~40-80 GB | ~1.5-2 TB/s | 10× slower |

Standard attention is **memory-bound** (I/O bound), not compute-bound:

1. **Write** $N \times N$ attention scores to HBM
2. **Read** them back for softmax
3. **Write** softmax result to HBM
4. **Read** for final matrix multiply

All these HBM accesses are the bottleneck!


In [ ]:
# Visualize the memory hierarchy
fig, ax = plt.subplots(figsize=(10, 6))

# Memory types
memories = ['SRAM\n(On-chip)', 'HBM\n(GPU Memory)', 'System RAM\n(CPU)']
sizes = [20, 80000, 128000]  # MB
bandwidths = [19000, 2000, 50]  # GB/s

x = np.arange(len(memories))
width = 0.35

# Normalize for visualization
sizes_norm = np.array(sizes) / max(sizes)
bw_norm = np.array(bandwidths) / max(bandwidths)

bars1 = ax.bar(x - width/2, bw_norm, width, label='Bandwidth (normalized)', color='#3498db')
bars2 = ax.bar(x + width/2, sizes_norm, width, label='Size (normalized)', color='#e74c3c')

ax.set_ylabel('Normalized Value', fontsize=12)
ax.set_title('GPU Memory Hierarchy: Speed vs Size Tradeoff', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(memories, fontsize=11)
ax.legend()

# Add actual values as text
for i, (bar1, bar2) in enumerate(zip(bars1, bars2)):
    ax.text(bar1.get_x() + bar1.get_width()/2, bar1.get_height() + 0.02,
            f'{bandwidths[i]} GB/s', ha='center', va='bottom', fontsize=9)
    ax.text(bar2.get_x() + bar2.get_width()/2, bar2.get_height() + 0.02,
            f'{sizes[i]} MB', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("Key insight: SRAM is 10× faster but 4000× smaller than HBM")
print("Flash Attention's goal: Keep data in SRAM as much as possible!")


## Part 3: The Flash Attention Algorithm

### Core Ideas

Flash Attention achieves O(N) memory by:

1. **Tiling**: Process attention in small blocks that fit in SRAM
2. **Online Softmax**: Compute softmax incrementally without materializing full matrix
3. **Recomputation**: In backward pass, recompute attention instead of storing it

### Online Softmax Trick

The key insight is that softmax can be computed **incrementally**:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}} = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}$$

where $m = \max(x)$ for numerical stability.

If we process blocks and get a new maximum, we can **rescale** previous results!


In [ ]:
def online_softmax_demo(x):
    """
    Demonstrate the online softmax algorithm.
    Shows how to compute softmax incrementally over blocks.
    """
    n = len(x)
    block_size = 2
    
    # Running statistics
    m = float('-inf')  # running max
    l = 0.0            # running sum of exp(x - m)
    
    # Process in blocks
    print("Processing blocks:")
    for i in range(0, n, block_size):
        block = x[i:i+block_size]
        
        # Get max of this block
        m_block = max(block)
        
        # Update running max
        m_new = max(m, m_block)
        
        # Rescale previous sum and add new terms
        # l_new = l * exp(m_old - m_new) + sum(exp(block - m_new))
        l = l * math.exp(m - m_new) + sum(math.exp(xi - m_new) for xi in block)
        m = m_new
        
        print(f"  Block {i//block_size}: {block.tolist()}, m={m:.2f}, l={l:.4f}")
    
    # Compute final softmax
    softmax_online = [math.exp(xi - m) / l for xi in x]
    
    # Compare with standard softmax
    softmax_standard = F.softmax(torch.tensor(x), dim=0).tolist()
    
    print(f"\nOnline softmax:   {[f'{v:.4f}' for v in softmax_online]}")
    print(f"Standard softmax: {[f'{v:.4f}' for v in softmax_standard]}")
    print(f"Match: {np.allclose(softmax_online, softmax_standard)}")
    
    return softmax_online

# Test online softmax
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
online_softmax_demo(x)


### The Tiling Strategy

Flash Attention divides Q, K, V into blocks and processes them tile by tile:

```
Q blocks: [Q_1, Q_2, Q_3, ...]
K blocks: [K_1, K_2, K_3, ...]
V blocks: [V_1, V_2, V_3, ...]

For each Q block:
    For each K, V block:
        Compute local attention
        Update running softmax statistics
        Accumulate output
```

This way, we never store the full N×N attention matrix!


In [ ]:
# Visualize the tiling strategy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Standard attention - full matrix
ax1 = axes[0]
n = 8
attention_matrix = np.random.rand(n, n)
sns.heatmap(attention_matrix, ax=ax1, cmap='YlOrRd', cbar=True,
            xticklabels=[f'K{i}' for i in range(n)],
            yticklabels=[f'Q{i}' for i in range(n)])
ax1.set_title('Standard: Materialize Full N×N Matrix', fontsize=12, fontweight='bold')
ax1.set_xlabel('Keys', fontsize=10)
ax1.set_ylabel('Queries', fontsize=10)

# Flash attention - tiled processing
ax2 = axes[1]
block_size = 2
tiled_matrix = np.zeros((n, n))

# Show only the current tile being processed
current_q_block = 1  # Processing Q block 1
current_k_block = 2  # Processing K block 2

# Highlight the tile being processed
for i in range(n):
    for j in range(n):
        q_block = i // block_size
        k_block = j // block_size
        if q_block == current_q_block and k_block == current_k_block:
            tiled_matrix[i, j] = 1.0  # Current tile
        elif q_block == current_q_block and k_block < current_k_block:
            tiled_matrix[i, j] = 0.3  # Already processed
        elif q_block < current_q_block:
            tiled_matrix[i, j] = 0.3  # Already processed

# Create custom colormap
cmap = plt.cm.colors.ListedColormap(['white', '#bdc3c7', '#e74c3c'])
bounds = [0, 0.15, 0.5, 1.1]
norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

sns.heatmap(tiled_matrix, ax=ax2, cmap=cmap, cbar=False,
            xticklabels=[f'K{i}' for i in range(n)],
            yticklabels=[f'Q{i}' for i in range(n)],
            linewidths=0.5, linecolor='gray')

# Add grid lines for blocks
for i in range(0, n+1, block_size):
    ax2.axhline(y=i, color='black', linewidth=2)
    ax2.axvline(x=i, color='black', linewidth=2)

ax2.set_title('Flash: Process One Tile at a Time', fontsize=12, fontweight='bold')
ax2.set_xlabel('Keys', fontsize=10)
ax2.set_ylabel('Queries', fontsize=10)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Currently Processing'),
    Patch(facecolor='#bdc3c7', label='Already Processed'),
    Patch(facecolor='white', edgecolor='gray', label='Not Yet Processed')
]
ax2.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Block size: {block_size}")
print(f"Memory for full matrix: {n*n} elements")
print(f"Memory for one tile: {block_size*block_size} elements")
print(f"Memory reduction: {(n*n) / (block_size*block_size):.1f}×")


In [ ]:
def flash_attention_simple(Q, K, V, block_size=32):
    """
    Simplified Flash Attention implementation.
    
    This is an educational implementation that demonstrates the core algorithm.
    Real Flash Attention uses optimized CUDA kernels.
    
    Args:
        Q: Queries (batch, seq_len, d_k)
        K: Keys (batch, seq_len, d_k)
        V: Values (batch, seq_len, d_k)
        block_size: Size of each tile
    
    Returns:
        output: Attention output (batch, seq_len, d_k)
    """
    batch_size, seq_len, d_k = Q.shape
    scale = 1.0 / math.sqrt(d_k)
    
    # Initialize output and softmax statistics
    O = torch.zeros_like(Q)
    L = torch.zeros(batch_size, seq_len, 1, device=Q.device)  # log-sum-exp
    M = torch.full((batch_size, seq_len, 1), float('-inf'), device=Q.device)  # row max
    
    # Number of blocks
    num_blocks = math.ceil(seq_len / block_size)
    
    # Process Q in blocks
    for q_block_idx in range(num_blocks):
        q_start = q_block_idx * block_size
        q_end = min(q_start + block_size, seq_len)
        
        Q_block = Q[:, q_start:q_end, :]  # (batch, block_size, d_k)
        
        # Running statistics for this Q block
        O_block = torch.zeros_like(Q_block)
        l_block = torch.zeros(batch_size, q_end - q_start, 1, device=Q.device)
        m_block = torch.full((batch_size, q_end - q_start, 1), float('-inf'), device=Q.device)
        
        # Process K, V in blocks
        for kv_block_idx in range(num_blocks):
            kv_start = kv_block_idx * block_size
            kv_end = min(kv_start + block_size, seq_len)
            
            K_block = K[:, kv_start:kv_end, :]  # (batch, block_size, d_k)
            V_block = V[:, kv_start:kv_end, :]  # (batch, block_size, d_k)
            
            # Compute attention scores for this tile
            # S = Q_block @ K_block.T * scale
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) * scale
            # Shape: (batch, q_block_size, kv_block_size)
            
            # Online softmax update
            # Step 1: Get new row max
            m_block_new = torch.max(m_block, S_block.max(dim=-1, keepdim=True).values)
            
            # Step 2: Compute exp(S - m_new) for this block
            P_block = torch.exp(S_block - m_block_new)
            
            # Step 3: Rescale previous output and accumulator
            # exp(m_old - m_new) is the rescaling factor
            rescale = torch.exp(m_block - m_block_new)
            
            # Step 4: Update running sum (denominator of softmax)
            l_block = l_block * rescale + P_block.sum(dim=-1, keepdim=True)
            
            # Step 5: Update output accumulator
            O_block = O_block * rescale + torch.matmul(P_block, V_block)
            
            # Step 6: Update max
            m_block = m_block_new
        
        # Normalize output for this Q block
        O[:, q_start:q_end, :] = O_block / l_block
    
    return O

print("Flash Attention implementation ready!")


### Verify Correctness

Let's verify that our Flash Attention produces the same results as standard attention.


## Part 5: Memory Comparison

### Theoretical Memory Analysis

| Algorithm | Attention Matrix Memory | Intermediate Memory |
|-----------|------------------------|---------------------|
| Standard  | O(N²)                  | O(N²)               |
| Flash     | O(B²) per tile         | O(N)                |

Where B is the block size (typically 64-256) and N is sequence length.


In [ ]:
def memory_analysis(seq_lengths, block_size=64, d_k=64, dtype_bytes=4):
    """
    Compare memory usage of standard vs flash attention.
    """
    standard_memory = []
    flash_memory = []
    
    for n in seq_lengths:
        # Standard: Need to store N×N attention matrix
        standard_mem = n * n * dtype_bytes
        standard_memory.append(standard_mem)
        
        # Flash: Only need to store:
        # - One tile of size B×B
        # - Running statistics (O, L, M) of size N×d_k, N×1, N×1
        tile_mem = block_size * block_size * dtype_bytes
        stats_mem = n * d_k * dtype_bytes + 2 * n * dtype_bytes
        flash_mem = tile_mem + stats_mem
        flash_memory.append(flash_mem)
    
    return standard_memory, flash_memory

# Analyze memory
seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
block_size = 64
standard_mem, flash_mem = memory_analysis(seq_lengths, block_size=block_size)

# Convert to MB
standard_mb = np.array(standard_mem) / (1024**2)
flash_mb = np.array(flash_mem) / (1024**2)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Memory usage
ax1 = axes[0]
ax1.semilogy(seq_lengths, standard_mb, 'o-', label='Standard Attention', 
             linewidth=2, markersize=8, color='#e74c3c')
ax1.semilogy(seq_lengths, flash_mb, 's-', label='Flash Attention', 
             linewidth=2, markersize=8, color='#27ae60')
ax1.set_xlabel('Sequence Length', fontsize=12)
ax1.set_ylabel('Memory (MB)', fontsize=12)
ax1.set_title('Memory Usage Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right plot: Memory savings ratio
ax2 = axes[1]
savings_ratio = np.array(standard_mem) / np.array(flash_mem)
ax2.bar(range(len(seq_lengths)), savings_ratio, color='#3498db')
ax2.set_xticks(range(len(seq_lengths)))
ax2.set_xticklabels(seq_lengths, rotation=45)
ax2.set_xlabel('Sequence Length', fontsize=12)
ax2.set_ylabel('Memory Savings (×)', fontsize=12)
ax2.set_title('Flash Attention Memory Savings', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (ratio, n) in enumerate(zip(savings_ratio, seq_lengths)):
    ax2.text(i, ratio + 1, f'{ratio:.0f}×', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nMemory comparison (block_size={block_size}):")
print(f"{'Seq Len':<10} {'Standard':<15} {'Flash':<15} {'Savings':<10}")
print("-" * 50)
for n, std, flash in zip(seq_lengths, standard_mb, flash_mb):
    print(f"{n:<10} {std:>10.2f} MB    {flash:>10.2f} MB    {std/flash:>6.0f}×")


## Part 6: Causal (Masked) Flash Attention

For autoregressive models (like GPT), we need **causal masking** where each position can only attend to previous positions.

Flash Attention handles this efficiently by:
1. Skipping tiles entirely if they're fully masked
2. Applying local masks only when necessary


In [ ]:
def flash_attention_causal(Q, K, V, block_size=32):
    """
    Flash Attention with causal masking.
    
    Each position can only attend to itself and previous positions.
    """
    batch_size, seq_len, d_k = Q.shape
    scale = 1.0 / math.sqrt(d_k)
    
    O = torch.zeros_like(Q)
    L = torch.zeros(batch_size, seq_len, 1, device=Q.device)
    M = torch.full((batch_size, seq_len, 1), float('-inf'), device=Q.device)
    
    num_blocks = math.ceil(seq_len / block_size)
    
    for q_block_idx in range(num_blocks):
        q_start = q_block_idx * block_size
        q_end = min(q_start + block_size, seq_len)
        
        Q_block = Q[:, q_start:q_end, :]
        
        O_block = torch.zeros_like(Q_block)
        l_block = torch.zeros(batch_size, q_end - q_start, 1, device=Q.device)
        m_block = torch.full((batch_size, q_end - q_start, 1), float('-inf'), device=Q.device)
        
        # Only process K,V blocks up to and including current Q block
        # (blocks after current Q block are fully masked)
        for kv_block_idx in range(q_block_idx + 1):
            kv_start = kv_block_idx * block_size
            kv_end = min(kv_start + block_size, seq_len)
            
            K_block = K[:, kv_start:kv_end, :]
            V_block = V[:, kv_start:kv_end, :]
            
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) * scale
            
            # Apply causal mask for diagonal blocks
            if q_block_idx == kv_block_idx:
                # Create local causal mask for this tile
                q_indices = torch.arange(q_start, q_end, device=Q.device).unsqueeze(1)
                k_indices = torch.arange(kv_start, kv_end, device=Q.device).unsqueeze(0)
                causal_mask = q_indices >= k_indices  # Lower triangular
                S_block = S_block.masked_fill(~causal_mask, float('-inf'))
            
            # Online softmax update (same as before)
            m_block_new = torch.max(m_block, S_block.max(dim=-1, keepdim=True).values)
            P_block = torch.exp(S_block - m_block_new)
            rescale = torch.exp(m_block - m_block_new)
            l_block = l_block * rescale + P_block.sum(dim=-1, keepdim=True)
            O_block = O_block * rescale + torch.matmul(P_block, V_block)
            m_block = m_block_new
        
        O[:, q_start:q_end, :] = O_block / l_block
    
    return O

print("Causal Flash Attention implementation ready!")


Verify the causal Flash Attention implementation against the standard masked baseline.

In [ ]:
# Verify causal flash attention
def standard_attention_causal(Q, K, V):
    """Standard attention with causal mask."""
    d_k = Q.size(-1)
    seq_len = Q.size(1)
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply causal mask
    causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=Q.device))
    scores = scores.masked_fill(causal_mask == 0, float('-inf'))
    
    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Test
print("Verifying Causal Flash Attention:")
print("=" * 60)

for seq_len, block_size in [(16, 4), (32, 8), (64, 16), (128, 32)]:
    Q = torch.randn(2, seq_len, 32)
    K = torch.randn(2, seq_len, 32)
    V = torch.randn(2, seq_len, 32)
    
    output_standard, _ = standard_attention_causal(Q, K, V)
    output_flash = flash_attention_causal(Q, K, V, block_size=block_size)
    
    max_diff = (output_standard - output_flash).abs().max().item()
    match = max_diff < 1e-5
    
    status = "✓" if match else "✗"
    print(f"{status} seq_len={seq_len:>4}, block_size={block_size:>3}: max_diff={max_diff:.2e}")

print("=" * 60)


### Visualize Causal Mask Efficiency

With causal masking, Flash Attention can skip about half the tiles!


In [ ]:
# Visualize which tiles are processed in causal attention
n = 8
block_size = 2
num_blocks = n // block_size

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Create tile matrix showing which tiles are computed
tile_matrix = np.zeros((num_blocks, num_blocks))

for q_block in range(num_blocks):
    for k_block in range(num_blocks):
        if k_block <= q_block:  # Only process tiles where K block <= Q block
            if k_block < q_block:
                tile_matrix[q_block, k_block] = 1.0  # Fully computed
            else:
                tile_matrix[q_block, k_block] = 0.5  # Diagonal (partial mask)

# Left: Tile-level view
ax1 = axes[0]
cmap = plt.cm.colors.ListedColormap(['white', '#f39c12', '#27ae60'])
bounds = [0, 0.25, 0.75, 1.1]
norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

sns.heatmap(tile_matrix, ax=ax1, cmap=cmap, cbar=False,
            xticklabels=[f'K[{i*block_size}:{(i+1)*block_size}]' for i in range(num_blocks)],
            yticklabels=[f'Q[{i*block_size}:{(i+1)*block_size}]' for i in range(num_blocks)],
            linewidths=2, linecolor='black')
ax1.set_title('Tile Processing in Causal Flash Attention', fontsize=12, fontweight='bold')
ax1.set_xlabel('K,V Blocks', fontsize=10)
ax1.set_ylabel('Q Blocks', fontsize=10)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#27ae60', label='Fully Computed'),
    Patch(facecolor='#f39c12', label='Partially Masked (Diagonal)'),
    Patch(facecolor='white', edgecolor='black', label='Skipped (Future)')
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=9)

# Right: Statistics
ax2 = axes[1]
total_tiles = num_blocks * num_blocks
computed_tiles = (tile_matrix > 0).sum()
skipped_tiles = total_tiles - computed_tiles

labels = ['Computed', 'Skipped']
sizes = [computed_tiles, skipped_tiles]
colors = ['#27ae60', '#e74c3c']
explode = (0.05, 0)

ax2.pie(sizes, labels=labels, colors=colors, explode=explode,
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
ax2.set_title(f'Tile Computation Savings\n({computed_tiles}/{total_tiles} tiles computed)', 
              fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"With causal masking:")
print(f"  Total tiles: {total_tiles}")
print(f"  Computed tiles: {computed_tiles}")
print(f"  Skipped tiles: {skipped_tiles}")
print(f"  Computation saved: {skipped_tiles/total_tiles*100:.1f}%")


## Part 7: Using Real Flash Attention

In practice, you should use optimized Flash Attention implementations:

### Option 1: PyTorch Native (2.0+)

```python
# PyTorch 2.0+ has built-in Flash Attention
from torch.nn.functional import scaled_dot_product_attention

output = scaled_dot_product_attention(Q, K, V, is_causal=True)
```

### Option 2: Flash Attention Library

```python
# pip install flash-attn
from flash_attn import flash_attn_func

output = flash_attn_func(Q, K, V, causal=True)
```


In [ ]:
# Test PyTorch's built-in scaled_dot_product_attention
print(f"PyTorch version: {torch.__version__}")

if hasattr(F, 'scaled_dot_product_attention'):
    print("✓ PyTorch scaled_dot_product_attention is available!")
    
    # Test it
    Q = torch.randn(2, 4, 64, 32)  # (batch, heads, seq_len, d_k)
    K = torch.randn(2, 4, 64, 32)
    V = torch.randn(2, 4, 64, 32)
    
    # Without causal mask
    output = F.scaled_dot_product_attention(Q, K, V)
    print(f"  Output shape: {output.shape}")
    
    # With causal mask
    output_causal = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    print(f"  Causal output shape: {output_causal.shape}")
    
    print("\n  PyTorch automatically selects the best backend:")
    print("  - Flash Attention (if available)")
    print("  - Memory-efficient attention")
    print("  - Standard attention (fallback)")
else:
    print("✗ PyTorch scaled_dot_product_attention not available (requires PyTorch 2.0+)")


## Part 8: Performance Characteristics

### When to Use Flash Attention

| Scenario | Benefit |
|----------|--------|
| Long sequences (N > 512) | Massive memory savings |
| Training large models | Lower memory = larger batch sizes |
| Limited GPU memory | Can handle sequences that wouldn't fit otherwise |
| Inference | Faster due to reduced memory bandwidth |

### Trade-offs

| Aspect | Standard | Flash |
|--------|----------|-------|
| Memory | O(N²) | O(N) |
| Compute | O(N²d) | O(N²d) (same!) |
| Attention weights | Stored | Recomputed in backward pass |
| Speed | I/O bound | Compute bound (faster!) |


In [ ]:
# Visualize when Flash Attention is most beneficial

# Theoretical speedup based on I/O vs compute ratio
# Standard attention: dominated by memory bandwidth
# Flash attention: dominated by compute

seq_lengths = np.array([128, 256, 512, 1024, 2048, 4096, 8192])
d_k = 64

# Simplified model of speedup (real speedup depends on hardware)
# Speedup increases with sequence length as memory becomes more of a bottleneck
theoretical_speedup = 1 + np.log2(seq_lengths / 128)
theoretical_speedup = np.clip(theoretical_speedup, 1, 4)  # Cap at 4x

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Theoretical speedup
ax1 = axes[0]
ax1.plot(seq_lengths, theoretical_speedup, 'o-', linewidth=2, markersize=8, color='#27ae60')
ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax1.fill_between(seq_lengths, 1, theoretical_speedup, alpha=0.3, color='#27ae60')
ax1.set_xlabel('Sequence Length', fontsize=12)
ax1.set_ylabel('Speedup vs Standard Attention', fontsize=12)
ax1.set_title('Flash Attention Speed Improvement', fontsize=14, fontweight='bold')
ax1.set_xscale('log', base=2)
ax1.grid(True, alpha=0.3)

# Right: Memory vs Compute comparison
ax2 = axes[1]

# For standard attention, time is dominated by memory I/O (O(N²))
# For flash attention, time is dominated by compute (O(N²d))
memory_io = seq_lengths ** 2  # Proportional to N²
compute = seq_lengths ** 2 * d_k  # Proportional to N²d

# Normalize for visualization
memory_io_norm = memory_io / memory_io.max()
compute_norm = compute / compute.max()

x = np.arange(len(seq_lengths))
width = 0.35

bars1 = ax2.bar(x - width/2, memory_io_norm, width, label='Memory I/O (Standard)', color='#e74c3c')
bars2 = ax2.bar(x + width/2, compute_norm, width, label='Compute (Flash)', color='#3498db')

ax2.set_xlabel('Sequence Length', fontsize=12)
ax2.set_ylabel('Relative Cost (normalized)', fontsize=12)
ax2.set_title('Bottleneck Comparison', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(seq_lengths)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key insight: Flash Attention converts attention from memory-bound to compute-bound")
print("This is better because modern GPUs have much more compute than memory bandwidth!")


## Part 9: The Backward Pass

### Recomputation Strategy

One key insight of Flash Attention is that it **doesn't store attention weights** for the backward pass. Instead, it **recomputes** them on the fly.

This is a classic **time-memory tradeoff**:
- Standard: Store attention weights → fast backward, high memory
- Flash: Recompute attention weights → slower backward, low memory

The recomputation overhead is small because:
1. Attention is computed tile-by-tile (fits in fast SRAM)
2. The forward pass is memory-bound anyway, so compute is "free"
3. Total memory saved is O(N²) which dominates the cost


In [ ]:
# Visualize the time-memory tradeoff
fig, ax = plt.subplots(figsize=(10, 6))

seq_lengths_plot = [512, 1024, 2048, 4096, 8192]

# Simplified model of memory and time
# Standard: high memory, moderate time
# Flash: low memory, slightly higher time (due to recomputation)

standard_memory = [n**2 / 1e6 for n in seq_lengths_plot]  # MB
flash_memory = [n * 64 / 1e6 for n in seq_lengths_plot]  # MB (O(N))

# Time relative to baseline
standard_time = [1.0] * len(seq_lengths_plot)
flash_time = [0.7, 0.6, 0.5, 0.45, 0.4]  # Flash is actually faster!

# Scatter plot
ax.scatter(standard_memory, standard_time, s=200, c='#e74c3c', marker='s', 
           label='Standard Attention', zorder=5)
ax.scatter(flash_memory, flash_time, s=200, c='#27ae60', marker='o', 
           label='Flash Attention', zorder=5)

# Connect corresponding points
for i, n in enumerate(seq_lengths_plot):
    ax.annotate('', xy=(flash_memory[i], flash_time[i]), 
                xytext=(standard_memory[i], standard_time[i]),
                arrowprops=dict(arrowstyle='->', color='gray', alpha=0.5))
    ax.annotate(f'N={n}', xy=(standard_memory[i], standard_time[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax.set_xlabel('Memory Usage (MB)', fontsize=12)
ax.set_ylabel('Relative Time', fontsize=12)
ax.set_title('Flash Attention: Better Memory AND Speed!', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Add region annotations
ax.annotate('Better\n(less memory, faster)', 
            xy=(0.1, 0.3), xycoords='axes fraction',
            fontsize=10, ha='center', color='#27ae60', fontweight='bold')

plt.tight_layout()
plt.show()

print("Counterintuitive result: Flash Attention is both more memory-efficient AND faster!")
print("This is because the memory bandwidth was the bottleneck, not compute.")


## Summary

### Key Takeaways

1. **The Problem**: Standard attention is O(N²) in memory and I/O-bound on GPUs

2. **GPU Memory Hierarchy**: 
   - SRAM is fast but small (~20 MB)
   - HBM is large but slow (~2 TB/s)
   - Standard attention wastes time moving data to/from HBM

3. **Flash Attention Solution**:
   - **Tiling**: Process attention in blocks that fit in SRAM
   - **Online Softmax**: Compute softmax incrementally using running max and sum
   - **Recomputation**: Don't store attention weights; recompute in backward pass

4. **Benefits**:
   - Memory: O(N²) → O(N)
   - Speed: 2-4× faster (memory bandwidth → compute bound)
   - Enables longer sequences and larger batch sizes

5. **How to Use**:
   - PyTorch 2.0+: `F.scaled_dot_product_attention(Q, K, V, is_causal=True)`
   - Flash Attention library: `flash_attn_func(Q, K, V, causal=True)`

### Further Reading

- [Flash Attention Paper](https://arxiv.org/abs/2205.14135) (Dao et al., 2022)
- [Flash Attention 2 Paper](https://arxiv.org/abs/2307.08691) (Dao, 2023)
- [Flash Attention GitHub](https://github.com/Dao-AILab/flash-attention)


In [ ]:
# Test with various sequence lengths and block sizes
test_configs = [
    (16, 4),   # seq_len=16, block_size=4
    (32, 8),   # seq_len=32, block_size=8
    (64, 16),  # seq_len=64, block_size=16
    (128, 32), # seq_len=128, block_size=32
]

print("Verifying Flash Attention correctness:")
print("=" * 60)

for seq_len, block_size in test_configs:
    batch_size = 2
    d_k = 32
    
    Q = torch.randn(batch_size, seq_len, d_k)
    K = torch.randn(batch_size, seq_len, d_k)
    V = torch.randn(batch_size, seq_len, d_k)
    
    # Standard attention
    output_standard, _ = standard_attention(Q, K, V)
    
    # Flash attention
    output_flash = flash_attention_simple(Q, K, V, block_size=block_size)
    
    # Check if they match
    max_diff = (output_standard - output_flash).abs().max().item()
    match = max_diff < 1e-5
    
    status = "✓" if match else "✗"
    print(f"{status} seq_len={seq_len:>4}, block_size={block_size:>3}: max_diff={max_diff:.2e}")

print("=" * 60)
print("Flash Attention produces correct results!")
